# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides an example of loading and exploring the FAIR² dataset using the `mlcroissant` library, following best practices for referencing data by entity `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (by their `@id`), fields, and columns.

In [ ]:
# Get a list of available record_set @id's
record_sets = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]

if record_sets:
    print('Record sets available (by @id):')
    for i, rs_id in enumerate(record_sets):
        print(f"  [{i}] {rs_id}")
else:
    # If not listed in metadata, introspect through dataset's interface
    print("Dataset metadata does not explicitly list record sets. Listing record set @id's from dataset:")
    all_record_sets = dataset.list_record_sets() if hasattr(dataset, 'list_record_sets') else []
    if all_record_sets:
        record_sets = all_record_sets
        for i, rs_id in enumerate(record_sets):
            print(f"  [{i}] {rs_id}")
    else:
        # Fallback: Try the records() generator with no record_set specified
        print('Exploring available records:')
        preview = list(dataset.records())
        if preview:
            example = preview[0]
            print('Example record:')
            print(example)
        else:
            print('No records found!')

For this dataset, record sets may not be present as explicit objects in the metadata. Let's enumerate all available record sets using the mlcroissant dataset API and inspect columns/fields using a records preview.

In [ ]:
# Attempt to obtain record_set ids via the mlcroissant API
# (As of mlcroissant 0.2.6+, Dataset.list_record_sets() yields the record_set @id's)
try:
    record_sets = dataset.list_record_sets()
except Exception:
    record_sets = []

print('Discovered record sets:')
for idx, rs_id in enumerate(record_sets):
    print(f"  [{idx}] {rs_id}")
    try:
        records_preview = list(dataset.records(record_set=rs_id))[:1]
        if records_preview:
            print(f"    Preview columns for {rs_id}: {list(records_preview[0].keys())}")
    except Exception:
        print("    Could not preview records.")
if not record_sets:
    print('No record sets found via API. Attempting to preview all records (flat):')
    flat_preview = list(dataset.records())
    if flat_preview:
        print(list(flat_preview[0].keys()))

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

As discovered above, we use the actual record set `@id`s.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

dataframes = dict()
if not record_sets:
    print('No record sets, extracting flat records:')
    records = list(dataset.records())
    if records:
        df = pd.DataFrame(records)
        dataframes['flat'] = df
        print(f"Extracted {len(records)} records.")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print('No records found.')
else:
    for rs_id in record_sets:
        print(f'Extracting records for record set @id: {rs_id}')
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f" - Loaded {df.shape[0]} rows, columns: {df.columns.tolist()}")
            print(df.head())
        except Exception as e:
            print(f" - Error: {str(e)}")

df_key = next(iter(dataframes.keys()))  # Choose first for further exploration
df = dataframes[df_key]

print(f'Using DataFrame for analysis: {df_key}')
print('Columns:')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing values, and grouping by key variables using their column (field) `@id`.

In [ ]:
# Identify a numeric field for example analysis
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use the first found numeric column
    print(f'Chosen numeric field: {numeric_field_id}')
else:
    print('No numeric field detected. Attempting with "log_likelihood" or "coefficient"...')
    numeric_field_id = None
    for col in df.columns:
        if 'log' in col or 'coef' in col or 'std' in col or 'p_' in col or 'value' in col:
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f'Fallback numeric field: {numeric_field_id}')
    else:
        print('No suitable numeric field found - EDA will be limited!')

if numeric_field_id:
    # Clean missing
    valid_numeric_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()].copy()
    valid_numeric_df[numeric_field_id] = pd.to_numeric(valid_numeric_df[numeric_field_id], errors='coerce')

    threshold = valid_numeric_df[numeric_field_id].mean() if valid_numeric_df.shape[0] > 0 else 0
    filtered_df = valid_numeric_df[valid_numeric_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (attempt 'variable', 'id', or similar)
    group_candidates = [col for col in df.columns if col not in numeric_fields and df[col].nunique() > 1 and df[col].nunique() < len(df)]
    group_field_id = group_candidates[0] if group_candidates else None
    if group_field_id:
        print(f'Grouping by field {group_field_id}')
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
    else:
        print('No suitable grouping field found.')
else:
    print('No EDA performed due to missing suitable numeric columns.')

## 5. Visualization

Visualize the distribution of the chosen numeric field and, if possible, compare across categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(valid_numeric_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        # Visualize by category using a boxplot
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load and examine the FAIR² dataset, inspected available record sets and fields by `@id`, and performed basic processing and visualization. Record sets and fields were referenced using `@id` throughout, ensuring full provenance traceability and clear mapping to the Croissant schema. For further analysis, consider deeper feature engineering and statistical modeling with this rich dataset on knowledge adoption in rangeland management in Kenya.